# 06 — Results Review

**Purpose:** Final review of flagged accounts — the analyst-facing notebook. This is what an AML analyst would open to understand which accounts were flagged and why.

This notebook is different from the others — it's less about validating the model and more about understanding the output as a human reviewer would. The questions we answer here:

- Which accounts are at the top of the anomaly list?
- What specific behaviors drove their scores? (Ranked feature contributions)
- What does the plain-English narrative say about each account?
- Did we catch the injected anomalies? Which patterns were hardest to detect?
- What do the supporting trades look like for the top-flagged accounts?

Run notebooks 01–05 first to generate all inputs.

In [ ]:
import sys
from pathlib import Path

import matplotlib
matplotlib.use("Agg")  # non-interactive backend
import matplotlib.pyplot as plt
%matplotlib inline
import pandas as pd
import seaborn as sns

sys.path.insert(0, str(Path.cwd().parent))

from aml_anomaly.reporting.flags import generate_narrative, get_ranked_features

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (12, 5)

DATA_DIR    = Path("../data")
MODELS_DIR  = Path("../models")
OUTPUTS_DIR = Path("../outputs")

In [ ]:
scores    = pd.read_csv(OUTPUTS_DIR / "anomaly_scores.csv")
accounts  = pd.read_csv(DATA_DIR / "raw" / "accounts.csv")
trades    = pd.read_csv(DATA_DIR / "holdout" / "new_trades.csv", parse_dates=["trade_date"])
features  = pd.read_csv(DATA_DIR / "features" / "feature_matrix.csv")
injected  = pd.read_csv(DATA_DIR / "raw" / "injected_anomalies.csv")

scores["is_injected"] = scores["account_id"].isin(injected["account_id"])
flagged = scores[scores["anomaly_flag"] == 1].sort_values("anomaly_rank")

print(f"Total accounts scored: {len(scores):,}")
print(f"Flagged accounts:      {len(flagged):,}")
print(f"Injected anomalies in flagged list: {flagged['is_injected'].sum()}")

## 1. Anomaly score overview

A histogram of all anomaly scores with the flagging threshold marked. We expect to see the population bunched near zero with a thin high-scoring tail.

In [ ]:
threshold = scores[scores["anomaly_flag"] == 1]["anomaly_score"].min()

fig, ax = plt.subplots(figsize=(12, 4))
ax.hist(scores["anomaly_score"], bins=80, edgecolor="none", alpha=0.8)
ax.axvline(threshold, color="crimson", linestyle="--", label=f"Flag threshold ({threshold:.3f})")
ax.set_xlabel("Ensemble Anomaly Score")
ax.set_ylabel("Number of Accounts")
ax.set_title("Distribution of Anomaly Scores — All Accounts")
ax.legend()
plt.tight_layout()
plt.show()

## 2. Top 20 flagged accounts

The accounts an analyst would review first. Red rows are injected anomalies — confirming the model is ranking them at the top.

In [ ]:
top20 = flagged.head(20).merge(
    accounts[["account_id", "customer_name", "account_type", "risk_tier", "is_pep", "country_of_origin"]],
    on="account_id"
).merge(
    injected[["account_id", "anomaly_pattern"]].rename(columns={"anomaly_pattern": "injected_pattern"}),
    on="account_id", how="left"
)

display_cols = ["anomaly_rank", "account_id", "customer_name", "account_type",
                "risk_tier", "anomaly_score", "is_pep", "injected_pattern"]

top20[display_cols].style.apply(
    lambda row: ["background-color: #ffe5e5" if row["injected_pattern"] is not None else "" for _ in row],
    axis=1
)

## 3. Plain-English narrative for a flagged account

This is the analyst-facing explanation — no variable names, no statistics jargon. Just a clear description of what the account was doing.

In [ ]:
# Change this to any account_id from the top flagged list above
example_account = flagged.iloc[0]["account_id"]

narrative = generate_narrative(example_account, features, scores)

print(f"Account: {example_account}")
print(f"Anomaly Rank: {scores.loc[scores['account_id'] == example_account, 'anomaly_rank'].values[0]}")
print("\nNarrative:")
print("-" * 60)
print(narrative)

## 4. Ranked feature contributions

The data-scientist-facing view: which features drove the score, how far from the population mean, and what was the account's actual value.

In [ ]:
feature_table = get_ranked_features(example_account, features, scores)
print(feature_table.to_string(index=False))

## 5. Supporting trades for the flagged account

The 20 most relevant individual transactions — the actual trades that drove the anomalous features.

In [ ]:
account_trades = trades[trades["account_id"] == example_account].sort_values("trade_date")

display_cols = [
    "trade_id", "trade_date", "trade_time", "ticker", "trade_direction",
    "quantity", "trade_value_usd", "counterparty_account_id", "is_off_hours", "is_round_value"
]
print(f"Total trades for {example_account}: {len(account_trades)}")
account_trades[display_cols].head(20)

## 6. Injected anomaly validation summary

The final validation: how many of the known injected anomalies did we catch, broken down by pattern? This is the qualitative benchmark for the whole system.

In [ ]:
validation = scores.merge(injected[["account_id", "anomaly_pattern"]], on="account_id", how="right")

summary = validation.groupby("anomaly_pattern").agg(
    total=("account_id", "count"),
    flagged=("anomaly_flag", "sum"),
    avg_score=("anomaly_score", "mean"),
    avg_rank=("anomaly_rank", "mean"),
).reset_index()
summary["recall"] = (summary["flagged"] / summary["total"]).map("{:.0%}".format)

print("Injected Anomaly Detection Summary:")
print(summary.to_string(index=False))

overall = validation["anomaly_flag"].mean()
print(f"\nOverall recall: {overall:.1%}")

## 7. Browse multiple flagged accounts

Interactive loop to view narratives and feature tables for any set of accounts.

In [ ]:
# Change n_accounts to review more or fewer accounts
n_accounts = 5

for _, row in flagged.head(n_accounts).iterrows():
    acct = row["account_id"]
    pattern = injected.loc[injected["account_id"] == acct, "anomaly_pattern"]
    pattern_str = f" [INJECTED: {pattern.values[0]}]" if len(pattern) > 0 else ""

    print(f"\n{'='*70}")
    print(f"Rank #{int(row['anomaly_rank'])} — {acct}{pattern_str}")
    print(f"Score: {row['anomaly_score']:.4f}")
    print("Narrative:")
    print(generate_narrative(acct, features, scores))
    print("\nTop features:")
    print(get_ranked_features(acct, features, scores).head(5).to_string(index=False))